In [ ]:
# Cell 1: Load the processed Ethiopia road-accident dataset and inspect driver-related variables and severity classes.

import pandas as pd
import numpy as np

ethiopia_df = pd.read_csv(
    "../data/processed/ethiopia_processed.csv"
)

driver_features = [
    "Age_band_of_driver",
    "Drivers_gender",
    "Educational_level",
    "Driving_experience",
    "Vehicle_driver_relation"
]

print("Ethiopia dataset shape:", ethiopia_df.shape)

print("\nDriver-related features:")
print(driver_features)

print("\nTarget distribution:")
print(ethiopia_df["severity_class"].value_counts(dropna=False))

print("\nTarget percentages:")
print(
    ethiopia_df["severity_class"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

print("\nDriver feature missing values:")
print(
    ethiopia_df[driver_features]
    .isna()
    .sum()
)

for col in driver_features:
    print("\n" + "=" * 70)
    print(col)
    print(ethiopia_df[col].value_counts(dropna=False))

In [ ]:
# Cell 2: Inspect available Ethiopia predictors and define candidate non-driver features before constructing matched comparison models.

print("All Ethiopia columns:")
for i, col in enumerate(ethiopia_df.columns, 1):
    print(f"{i:02d}. {col}")

print("\nDataset shape:", ethiopia_df.shape)

print("\nData types:")
print(ethiopia_df.dtypes)

print("\nMissing values:")
print(
    ethiopia_df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
# Cell 3: Define matched Ethiopia feature sets for comparing accident/environment predictors with and without driver information.

base_features = [
    "hour",
    "day_common",
    "weather_common",
    "light_common",
    "surface_common",
    "junction_common",
    "vehicle_category",
    "vehicle_count",
    "has_motorcycle",
    "has_heavy_vehicle",
    "has_public_transport",
    "has_two_wheeler",
    "Type_of_collision",
    "Number_of_casualties",
    "Road_surface_type",
    "Road_allignment",
    "Area_accident_occured",
    "Vehicle_movement",
    "Cause_of_accident"
]

driver_features = [
    "Age_band_of_driver",
    "Drivers_gender",
    "Educational_level",
    "Driving_experience",
    "Vehicle_driver_relation"
]

target = "severity_class"

model_a_features = base_features
model_b_features = base_features + driver_features

ethiopia_model_df = ethiopia_df[
    model_b_features + [target]
].copy()

categorical_features_all = [
    col for col in model_b_features
    if ethiopia_model_df[col].dtype == "object"
]

# Preserve missing categorical information instead of dropping rows
for col in categorical_features_all:
    ethiopia_model_df[col] = (
        ethiopia_model_df[col]
        .fillna("Missing")
        .astype(str)
    )

print("Model A feature count:", len(model_a_features))
print("Model B feature count:", len(model_b_features))

print("\nDriver features added in Model B:")
print(driver_features)

print("\nDataset retained after preparation:", ethiopia_model_df.shape)

print("\nRemaining missing values:")
print(ethiopia_model_df.isna().sum().sum())

print("\nTarget distribution:")
print(ethiopia_model_df[target].value_counts())

In [ ]:
# Cell 4: Create one identical stratified 80/20 train-test split for both Ethiopia models.

from sklearn.model_selection import train_test_split

# Use row indices so Models A and B receive exactly the same cases
train_idx, test_idx = train_test_split(
    ethiopia_model_df.index,
    test_size=0.20,
    random_state=42,
    stratify=ethiopia_model_df[target]
)

# Model A — accident/environment features only
X_train_A = ethiopia_model_df.loc[
    train_idx, model_a_features
].copy()

X_test_A = ethiopia_model_df.loc[
    test_idx, model_a_features
].copy()

# Model B — same features + driver information
X_train_B = ethiopia_model_df.loc[
    train_idx, model_b_features
].copy()

X_test_B = ethiopia_model_df.loc[
    test_idx, model_b_features
].copy()

# Identical targets for both models
y_train_eth = ethiopia_model_df.loc[
    train_idx, target
].copy()

y_test_eth = ethiopia_model_df.loc[
    test_idx, target
].copy()

print("Training samples:", len(train_idx))
print("Test samples:", len(test_idx))

print("\nTraining class distribution:")
print(y_train_eth.value_counts())

print("\nTraining percentages:")
print(
    y_train_eth.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTest class distribution:")
print(y_test_eth.value_counts())

print("\nTest percentages:")
print(
    y_test_eth.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nModel A:")
print("Train:", X_train_A.shape)
print("Test:", X_test_A.shape)

print("\nModel B:")
print("Train:", X_train_B.shape)
print("Test:", X_test_B.shape)

print(
    "\nIdentical target cases for A and B:",
    X_train_A.index.equals(X_train_B.index)
    and X_test_A.index.equals(X_test_B.index)
)

In [ ]:
# Cell 5: Train and evaluate baseline CatBoost Model A using accident/environment features only, without driver information.

from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

categorical_A = [
    col for col in model_a_features
    if X_train_A[col].dtype == "object"
]

model_A = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    verbose=False
)

model_A.fit(
    X_train_A,
    y_train_eth,
    cat_features=categorical_A
)

y_pred_A = model_A.predict(X_test_A).ravel()

accuracy_A = accuracy_score(
    y_test_eth, y_pred_A
)

macro_precision_A = precision_score(
    y_test_eth,
    y_pred_A,
    average="macro",
    zero_division=0
)

macro_recall_A = recall_score(
    y_test_eth,
    y_pred_A,
    average="macro",
    zero_division=0
)

macro_f1_A = f1_score(
    y_test_eth,
    y_pred_A,
    average="macro",
    zero_division=0
)

print("Ethiopia Model A — Accident/Environment Features Only")
print("-" * 65)

print(f"Accuracy:        {accuracy_A:.4f}")
print(f"Macro Precision: {macro_precision_A:.4f}")
print(f"Macro Recall:    {macro_recall_A:.4f}")
print(f"Macro F1-score:  {macro_f1_A:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test_eth,
        y_pred_A,
        labels=["Fatal", "Serious", "Slight"],
        digits=4,
        zero_division=0
    )
)

cm_A = confusion_matrix(
    y_test_eth,
    y_pred_A,
    labels=["Fatal", "Serious", "Slight"]
)

cm_A_df = pd.DataFrame(
    cm_A,
    index=[
        "Actual_Fatal",
        "Actual_Serious",
        "Actual_Slight"
    ],
    columns=[
        "Predicted_Fatal",
        "Predicted_Serious",
        "Predicted_Slight"
    ]
)

print("\nConfusion Matrix:")
print(cm_A_df)

In [ ]:
# Cell 6: Train and evaluate CatBoost Model B using the same accident/environment features plus five driver-related features.

categorical_B = [
    col for col in model_b_features
    if X_train_B[col].dtype == "object"
]

model_B = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    verbose=False
)

model_B.fit(
    X_train_B,
    y_train_eth,
    cat_features=categorical_B
)

y_pred_B = model_B.predict(X_test_B).ravel()

accuracy_B = accuracy_score(
    y_test_eth, y_pred_B
)

macro_precision_B = precision_score(
    y_test_eth,
    y_pred_B,
    average="macro",
    zero_division=0
)

macro_recall_B = recall_score(
    y_test_eth,
    y_pred_B,
    average="macro",
    zero_division=0
)

macro_f1_B = f1_score(
    y_test_eth,
    y_pred_B,
    average="macro",
    zero_division=0
)

print("Ethiopia Model B — Accident/Environment + Driver Features")
print("-" * 70)

print(f"Accuracy:        {accuracy_B:.4f}")
print(f"Macro Precision: {macro_precision_B:.4f}")
print(f"Macro Recall:    {macro_recall_B:.4f}")
print(f"Macro F1-score:  {macro_f1_B:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test_eth,
        y_pred_B,
        labels=["Fatal", "Serious", "Slight"],
        digits=4,
        zero_division=0
    )
)

cm_B = confusion_matrix(
    y_test_eth,
    y_pred_B,
    labels=["Fatal", "Serious", "Slight"]
)

cm_B_df = pd.DataFrame(
    cm_B,
    index=[
        "Actual_Fatal",
        "Actual_Serious",
        "Actual_Slight"
    ],
    columns=[
        "Predicted_Fatal",
        "Predicted_Serious",
        "Predicted_Slight"
    ]
)

print("\nConfusion Matrix:")
print(cm_B_df)

print("\nModel A vs Model B")
print("-" * 45)

print(f"Accuracy:        {accuracy_A:.4f} -> {accuracy_B:.4f}")
print(f"Macro Precision: {macro_precision_A:.4f} -> {macro_precision_B:.4f}")
print(f"Macro Recall:    {macro_recall_A:.4f} -> {macro_recall_B:.4f}")
print(f"Macro F1:        {macro_f1_A:.4f} -> {macro_f1_B:.4f}")

In [ ]:
# Cell 7: Compare Models A and B again using identical balanced class weights to test whether driver information helps minority-class prediction.

# Calculate balanced class weights from Ethiopia training data
eth_class_counts = y_train_eth.value_counts()

eth_total = len(y_train_eth)
eth_n_classes = len(eth_class_counts)

eth_class_weights = {
    cls: eth_total / (eth_n_classes * count)
    for cls, count in eth_class_counts.items()
}

print("Ethiopia balanced class weights:")
for cls, weight in eth_class_weights.items():
    print(f"{cls}: {weight:.4f}")

# Weighted Model A
weighted_model_A = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    verbose=False,
    class_weights=eth_class_weights
)

weighted_model_A.fit(
    X_train_A,
    y_train_eth,
    cat_features=categorical_A
)

y_pred_A_w = weighted_model_A.predict(X_test_A).ravel()

# Weighted Model B
weighted_model_B = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    verbose=False,
    class_weights=eth_class_weights
)

weighted_model_B.fit(
    X_train_B,
    y_train_eth,
    cat_features=categorical_B
)

y_pred_B_w = weighted_model_B.predict(X_test_B).ravel()

# Evaluation
def get_metrics(y_true, y_pred):
    report = classification_report(
        y_true,
        y_pred,
        labels=["Fatal", "Serious", "Slight"],
        output_dict=True,
        zero_division=0
    )
    
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro_F1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Macro_Recall": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Fatal_Recall": report["Fatal"]["recall"],
        "Serious_Recall": report["Serious"]["recall"],
        "Slight_Recall": report["Slight"]["recall"]
    }

metrics_A_w = get_metrics(y_test_eth, y_pred_A_w)
metrics_B_w = get_metrics(y_test_eth, y_pred_B_w)

comparison_weighted = pd.DataFrame(
    [metrics_A_w, metrics_B_w],
    index=[
        "Model A Weighted — No Driver Features",
        "Model B Weighted — With Driver Features"
    ]
).round(4)

print("\nWeighted Model Comparison")
print(comparison_weighted.to_string())

In [ ]:
# Cell 7: Train and compare class-weighted CatBoost models with and without driver-related features on the Ethiopia dataset.

# Weighted Model A — accident/environment features only
weighted_model_A = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    verbose=False,
    class_weights=eth_class_weights
)

weighted_model_A.fit(
    X_train_A,
    y_train_eth,
    cat_features=categorical_A
)

y_pred_A_w = weighted_model_A.predict(X_test_A).ravel()


# Weighted Model B — accident/environment + driver features
weighted_model_B = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    verbose=False,
    class_weights=eth_class_weights
)

weighted_model_B.fit(
    X_train_B,
    y_train_eth,
    cat_features=categorical_B
)

y_pred_B_w = weighted_model_B.predict(X_test_B).ravel()


# Evaluation function
def evaluate_weighted_model(y_true, y_pred):
    report = classification_report(
        y_true,
        y_pred,
        labels=["Fatal", "Serious", "Slight"],
        output_dict=True,
        zero_division=0
    )

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro_Precision": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "Macro_Recall": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "Macro_F1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "Fatal_Precision": report["Fatal"]["precision"],
        "Fatal_Recall": report["Fatal"]["recall"],
        "Fatal_F1": report["Fatal"]["f1-score"],
        "Serious_Precision": report["Serious"]["precision"],
        "Serious_Recall": report["Serious"]["recall"],
        "Serious_F1": report["Serious"]["f1-score"],
        "Slight_Recall": report["Slight"]["recall"]
    }


metrics_A_w = evaluate_weighted_model(
    y_test_eth,
    y_pred_A_w
)

metrics_B_w = evaluate_weighted_model(
    y_test_eth,
    y_pred_B_w
)

comparison_weighted = pd.DataFrame(
    [metrics_A_w, metrics_B_w],
    index=[
        "Model A Weighted — No Driver Features",
        "Model B Weighted — With Driver Features"
    ]
).round(4)

print("Weighted Ethiopia Model Comparison")
print("-" * 90)
print(comparison_weighted.to_string())

print("\nModel A Weighted Classification Report:")
print(
    classification_report(
        y_test_eth,
        y_pred_A_w,
        labels=["Fatal", "Serious", "Slight"],
        digits=4,
        zero_division=0
    )
)

print("\nModel B Weighted Classification Report:")
print(
    classification_report(
        y_test_eth,
        y_pred_B_w,
        labels=["Fatal", "Serious", "Slight"],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# Cell 8: Perform driver-feature ablation analysis by adding each driver variable separately to the weighted baseline model.

driver_ablation_results = []

# Baseline result without driver features
driver_ablation_results.append({
    "Added_Driver_Feature": "None (Baseline)",
    "Accuracy": metrics_A_w["Accuracy"],
    "Macro_F1": metrics_A_w["Macro_F1"],
    "Macro_Recall": metrics_A_w["Macro_Recall"],
    "Fatal_Recall": metrics_A_w["Fatal_Recall"],
    "Fatal_F1": metrics_A_w["Fatal_F1"],
    "Serious_Recall": metrics_A_w["Serious_Recall"],
    "Serious_F1": metrics_A_w["Serious_F1"],
    "Slight_Recall": metrics_A_w["Slight_Recall"]
})

for driver_feature in driver_features:

    current_features = model_a_features + [driver_feature]

    X_train_current = ethiopia_model_df.loc[
        train_idx, current_features
    ].copy()

    X_test_current = ethiopia_model_df.loc[
        test_idx, current_features
    ].copy()

    categorical_current = [
        col for col in current_features
        if X_train_current[col].dtype == "object"
    ]

    current_model = CatBoostClassifier(
        iterations=500,
        depth=8,
        learning_rate=0.05,
        loss_function="MultiClass",
        random_seed=42,
        verbose=False,
        class_weights=eth_class_weights
    )

    current_model.fit(
        X_train_current,
        y_train_eth,
        cat_features=categorical_current
    )

    current_pred = current_model.predict(
        X_test_current
    ).ravel()

    current_report = classification_report(
        y_test_eth,
        current_pred,
        labels=["Fatal", "Serious", "Slight"],
        output_dict=True,
        zero_division=0
    )

    driver_ablation_results.append({
        "Added_Driver_Feature": driver_feature,

        "Accuracy": accuracy_score(
            y_test_eth,
            current_pred
        ),

        "Macro_F1": f1_score(
            y_test_eth,
            current_pred,
            average="macro",
            zero_division=0
        ),

        "Macro_Recall": recall_score(
            y_test_eth,
            current_pred,
            average="macro",
            zero_division=0
        ),

        "Fatal_Recall":
            current_report["Fatal"]["recall"],

        "Fatal_F1":
            current_report["Fatal"]["f1-score"],

        "Serious_Recall":
            current_report["Serious"]["recall"],

        "Serious_F1":
            current_report["Serious"]["f1-score"],

        "Slight_Recall":
            current_report["Slight"]["recall"]
    })

driver_ablation_df = pd.DataFrame(
    driver_ablation_results
).round(4)

driver_ablation_df["Macro_F1_Change"] = (
    driver_ablation_df["Macro_F1"]
    - metrics_A_w["Macro_F1"]
).round(4)

driver_ablation_df = driver_ablation_df.sort_values(
    "Macro_F1",
    ascending=False
)

print("Driver Feature Ablation Analysis — Ethiopia")
print("-" * 100)

print(
    driver_ablation_df.to_string(index=False)
)

In [ ]:
# Cell 9: Examine observed accident-severity distributions across driver demographic and experience groups.

driver_severity_tables = {}

for feature in driver_features:

    counts = pd.crosstab(
        ethiopia_model_df[feature],
        ethiopia_model_df["severity_class"]
    ).reindex(
        columns=["Slight", "Serious", "Fatal"],
        fill_value=0
    )

    counts["Total"] = counts.sum(axis=1)

    percentages = (
        counts[["Slight", "Serious", "Fatal"]]
        .div(counts["Total"], axis=0)
        .mul(100)
        .round(2)
    )

    percentages["Total"] = counts["Total"]

    driver_severity_tables[feature] = percentages

    print("\n" + "=" * 85)
    print(f"Severity Distribution by {feature} (%)")
    print("=" * 85)

    print(
        percentages
        .sort_values("Total", ascending=False)
        .to_string()
    )

In [ ]:
# Cell 10: Rank the predictive importance of accident, road, environment, and vehicle features in the best weighted Ethiopia Model A.

feature_importance_A = pd.DataFrame({
    "Feature": model_a_features,
    "Importance": weighted_model_A.get_feature_importance()
})

feature_importance_A = (
    feature_importance_A
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance_A["Importance_Percent"] = (
    feature_importance_A["Importance"]
    / feature_importance_A["Importance"].sum()
    * 100
).round(2)

feature_importance_A["Cumulative_Percent"] = (
    feature_importance_A["Importance_Percent"]
    .cumsum()
    .round(2)
)

print("Weighted Ethiopia Model A — Feature Importance")
print("-" * 75)

print(
    feature_importance_A.to_string(
        index=False
    )
)

print("\nTop 10 features:")
print(
    feature_importance_A[
        ["Feature", "Importance_Percent"]
    ]
    .head(10)
    .to_string(index=False)
)

In [ ]:
# Cell 11: Rank all accident/environment and driver-related features in the weighted Ethiopia Model B.

feature_importance_B = pd.DataFrame({
    "Feature": model_b_features,
    "Importance": weighted_model_B.get_feature_importance()
})

feature_importance_B = (
    feature_importance_B
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance_B["Importance_Percent"] = (
    feature_importance_B["Importance"]
    / feature_importance_B["Importance"].sum()
    * 100
).round(2)

feature_importance_B["Rank"] = (
    np.arange(1, len(feature_importance_B) + 1)
)

feature_importance_B["Feature_Group"] = np.where(
    feature_importance_B["Feature"].isin(driver_features),
    "Driver",
    "Accident/Environment"
)

print("Weighted Ethiopia Model B — Full Feature Importance")
print("-" * 90)

print(
    feature_importance_B[
        [
            "Rank",
            "Feature",
            "Feature_Group",
            "Importance_Percent"
        ]
    ].to_string(index=False)
)

print("\nDriver features only:")
print(
    feature_importance_B[
        feature_importance_B["Feature_Group"] == "Driver"
    ][
        [
            "Rank",
            "Feature",
            "Importance_Percent"
        ]
    ].to_string(index=False)
)

driver_total_importance = (
    feature_importance_B.loc[
        feature_importance_B["Feature_Group"] == "Driver",
        "Importance_Percent"
    ].sum()
)

environment_total_importance = (
    feature_importance_B.loc[
        feature_importance_B["Feature_Group"] == "Accident/Environment",
        "Importance_Percent"
    ].sum()
)

print("\nTotal driver-feature importance:",
      round(driver_total_importance, 2), "%")

print("Total accident/environment importance:",
      round(environment_total_importance, 2), "%")

In [ ]:
# Cell 12: Train a driver-only weighted CatBoost model to measure the standalone predictive value of driver characteristics.

X_train_C = ethiopia_model_df.loc[
    train_idx, driver_features
].copy()

X_test_C = ethiopia_model_df.loc[
    test_idx, driver_features
].copy()

categorical_C = driver_features.copy()

model_C = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    verbose=False,
    class_weights=eth_class_weights
)

model_C.fit(
    X_train_C,
    y_train_eth,
    cat_features=categorical_C
)

y_pred_C = model_C.predict(X_test_C).ravel()

report_C = classification_report(
    y_test_eth,
    y_pred_C,
    labels=["Fatal", "Serious", "Slight"],
    output_dict=True,
    zero_division=0
)

metrics_C = {
    "Accuracy": accuracy_score(y_test_eth, y_pred_C),

    "Macro_Precision": precision_score(
        y_test_eth,
        y_pred_C,
        average="macro",
        zero_division=0
    ),

    "Macro_Recall": recall_score(
        y_test_eth,
        y_pred_C,
        average="macro",
        zero_division=0
    ),

    "Macro_F1": f1_score(
        y_test_eth,
        y_pred_C,
        average="macro",
        zero_division=0
    ),

    "Fatal_Recall": report_C["Fatal"]["recall"],
    "Fatal_F1": report_C["Fatal"]["f1-score"],

    "Serious_Recall": report_C["Serious"]["recall"],
    "Serious_F1": report_C["Serious"]["f1-score"],

    "Slight_Recall": report_C["Slight"]["recall"]
}

print("Ethiopia Model C — Driver Features Only")
print("-" * 65)

print(
    classification_report(
        y_test_eth,
        y_pred_C,
        labels=["Fatal", "Serious", "Slight"],
        digits=4,
        zero_division=0
    )
)

print("\nOverall Metrics:")
for metric, value in metrics_C.items():
    print(f"{metric}: {value:.4f}")


# Final A-B-C comparison

final_abc_comparison = pd.DataFrame(
    [
        {
            "Model": "A — Accident/Environment Only",
            **metrics_A_w
        },
        {
            "Model": "B — Accident/Environment + Driver",
            **metrics_B_w
        },
        {
            "Model": "C — Driver Only",
            **metrics_C
        }
    ]
)

selected_columns = [
    "Model",
    "Accuracy",
    "Macro_F1",
    "Macro_Recall",
    "Fatal_Recall",
    "Fatal_F1",
    "Serious_Recall",
    "Serious_F1",
    "Slight_Recall"
]

print("\nFinal A-B-C Comparison")
print("-" * 100)

print(
    final_abc_comparison[
        selected_columns
    ].round(4).to_string(index=False)
)

In [ ]:
# Cell 13: Compare Models A and B using 5-fold repeated stratified cross-validation with fold-specific class weights.

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

X_A_full = ethiopia_model_df[model_a_features].copy()
X_B_full = ethiopia_model_df[model_b_features].copy()
y_full = ethiopia_model_df[target].copy()

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=42
)

cv_results = []

for fold_number, (train_indices, test_indices) in enumerate(
    cv.split(X_A_full, y_full),
    start=1
):

    y_train_fold = y_full.iloc[train_indices]
    y_test_fold = y_full.iloc[test_indices]

    # Calculate class weights using TRAINING FOLD ONLY
    fold_counts = y_train_fold.value_counts()
    fold_total = len(y_train_fold)
    fold_n_classes = len(fold_counts)

    fold_weights = {
        cls: fold_total / (fold_n_classes * count)
        for cls, count in fold_counts.items()
    }

    # =========================================================
    # Model A — Accident/Environment only
    # =========================================================

    X_train_A_fold = X_A_full.iloc[train_indices]
    X_test_A_fold = X_A_full.iloc[test_indices]

    model_A_fold = CatBoostClassifier(
        iterations=500,
        depth=8,
        learning_rate=0.05,
        loss_function="MultiClass",
        random_seed=42,
        verbose=False,
        class_weights=fold_weights
    )

    model_A_fold.fit(
        X_train_A_fold,
        y_train_fold,
        cat_features=categorical_A
    )

    pred_A_fold = model_A_fold.predict(
        X_test_A_fold
    ).ravel()

    report_A = classification_report(
        y_test_fold,
        pred_A_fold,
        labels=["Fatal", "Serious", "Slight"],
        output_dict=True,
        zero_division=0
    )

    cv_results.append({
        "Fold": fold_number,
        "Model": "A_No_Driver",
        "Accuracy": accuracy_score(
            y_test_fold, pred_A_fold
        ),
        "Macro_Precision": precision_score(
            y_test_fold,
            pred_A_fold,
            average="macro",
            zero_division=0
        ),
        "Macro_Recall": recall_score(
            y_test_fold,
            pred_A_fold,
            average="macro",
            zero_division=0
        ),
        "Macro_F1": f1_score(
            y_test_fold,
            pred_A_fold,
            average="macro",
            zero_division=0
        ),
        "Fatal_Recall": report_A["Fatal"]["recall"],
        "Fatal_F1": report_A["Fatal"]["f1-score"],
        "Serious_Recall": report_A["Serious"]["recall"],
        "Serious_F1": report_A["Serious"]["f1-score"],
        "Slight_Recall": report_A["Slight"]["recall"]
    })

    # =========================================================
    # Model B — Accident/Environment + Driver
    # =========================================================

    X_train_B_fold = X_B_full.iloc[train_indices]
    X_test_B_fold = X_B_full.iloc[test_indices]

    model_B_fold = CatBoostClassifier(
        iterations=500,
        depth=8,
        learning_rate=0.05,
        loss_function="MultiClass",
        random_seed=42,
        verbose=False,
        class_weights=fold_weights
    )

    model_B_fold.fit(
        X_train_B_fold,
        y_train_fold,
        cat_features=categorical_B
    )

    pred_B_fold = model_B_fold.predict(
        X_test_B_fold
    ).ravel()

    report_B = classification_report(
        y_test_fold,
        pred_B_fold,
        labels=["Fatal", "Serious", "Slight"],
        output_dict=True,
        zero_division=0
    )

    cv_results.append({
        "Fold": fold_number,
        "Model": "B_With_Driver",
        "Accuracy": accuracy_score(
            y_test_fold, pred_B_fold
        ),
        "Macro_Precision": precision_score(
            y_test_fold,
            pred_B_fold,
            average="macro",
            zero_division=0
        ),
        "Macro_Recall": recall_score(
            y_test_fold,
            pred_B_fold,
            average="macro",
            zero_division=0
        ),
        "Macro_F1": f1_score(
            y_test_fold,
            pred_B_fold,
            average="macro",
            zero_division=0
        ),
        "Fatal_Recall": report_B["Fatal"]["recall"],
        "Fatal_F1": report_B["Fatal"]["f1-score"],
        "Serious_Recall": report_B["Serious"]["recall"],
        "Serious_F1": report_B["Serious"]["f1-score"],
        "Slight_Recall": report_B["Slight"]["recall"]
    })

    print(f"Completed fold {fold_number}/15")


cv_results_df = pd.DataFrame(cv_results)

print("\nRepeated Stratified Cross-Validation completed.")
print("Total evaluations:", len(cv_results_df))

In [ ]:
# Cell 14: Summarize repeated cross-validation performance for Models A and B.

metrics_to_summarize = [
    "Accuracy",
    "Macro_Precision",
    "Macro_Recall",
    "Macro_F1",
    "Fatal_Recall",
    "Fatal_F1",
    "Serious_Recall",
    "Serious_F1",
    "Slight_Recall"
]

cv_summary = (
    cv_results_df
    .groupby("Model")[metrics_to_summarize]
    .agg(["mean", "std"])
    .round(4)
)

print("Repeated Stratified CV — Mean ± SD")
print("=" * 100)
print(cv_summary.to_string())


# Paired fold-by-fold difference: B - A

A_cv = (
    cv_results_df[
        cv_results_df["Model"] == "A_No_Driver"
    ]
    .sort_values("Fold")
    .reset_index(drop=True)
)

B_cv = (
    cv_results_df[
        cv_results_df["Model"] == "B_With_Driver"
    ]
    .sort_values("Fold")
    .reset_index(drop=True)
)

paired_difference = pd.DataFrame({
    "Fold": A_cv["Fold"],
    "Macro_F1_A": A_cv["Macro_F1"],
    "Macro_F1_B": B_cv["Macro_F1"]
})

paired_difference["Delta_B_minus_A"] = (
    paired_difference["Macro_F1_B"]
    - paired_difference["Macro_F1_A"]
)

print("\nMacro-F1 — Fold-by-Fold Comparison")
print("-" * 65)
print(
    paired_difference.round(4).to_string(index=False)
)

print("\nMean Macro-F1 difference (B - A):")
print(
    round(
        paired_difference["Delta_B_minus_A"].mean(),
        4
    )
)

print("\nNumber of folds where Model B was better:")
print(
    (
        paired_difference["Delta_B_minus_A"] > 0
    ).sum(),
    "/",
    len(paired_difference)
)

In [ ]:
# Cell 15: Evaluate the driver-only Model C using the same 5x3 repeated stratified cross-validation splits.

X_C_full = ethiopia_model_df[driver_features].copy()

cv_results_C = []

# Recreate exactly the same CV splitter
cv_C = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=42
)

for fold_number, (train_indices, test_indices) in enumerate(
    cv_C.split(X_C_full, y_full),
    start=1
):

    X_train_C_fold = X_C_full.iloc[train_indices]
    X_test_C_fold = X_C_full.iloc[test_indices]

    y_train_fold = y_full.iloc[train_indices]
    y_test_fold = y_full.iloc[test_indices]

    # Calculate weights from training fold only
    fold_counts = y_train_fold.value_counts()
    fold_total = len(y_train_fold)
    fold_n_classes = len(fold_counts)

    fold_weights = {
        cls: fold_total / (fold_n_classes * count)
        for cls, count in fold_counts.items()
    }

    model_C_fold = CatBoostClassifier(
        iterations=500,
        depth=8,
        learning_rate=0.05,
        loss_function="MultiClass",
        random_seed=42,
        verbose=False,
        class_weights=fold_weights
    )

    model_C_fold.fit(
        X_train_C_fold,
        y_train_fold,
        cat_features=driver_features
    )

    pred_C_fold = model_C_fold.predict(
        X_test_C_fold
    ).ravel()

    report_C_fold = classification_report(
        y_test_fold,
        pred_C_fold,
        labels=["Fatal", "Serious", "Slight"],
        output_dict=True,
        zero_division=0
    )

    cv_results_C.append({
        "Fold": fold_number,
        "Model": "C_Driver_Only",

        "Accuracy": accuracy_score(
            y_test_fold, pred_C_fold
        ),

        "Macro_Precision": precision_score(
            y_test_fold,
            pred_C_fold,
            average="macro",
            zero_division=0
        ),

        "Macro_Recall": recall_score(
            y_test_fold,
            pred_C_fold,
            average="macro",
            zero_division=0
        ),

        "Macro_F1": f1_score(
            y_test_fold,
            pred_C_fold,
            average="macro",
            zero_division=0
        ),

        "Fatal_Recall":
            report_C_fold["Fatal"]["recall"],

        "Fatal_F1":
            report_C_fold["Fatal"]["f1-score"],

        "Serious_Recall":
            report_C_fold["Serious"]["recall"],

        "Serious_F1":
            report_C_fold["Serious"]["f1-score"],

        "Slight_Recall":
            report_C_fold["Slight"]["recall"]
    })

    print(f"Completed fold {fold_number}/15")


cv_results_C_df = pd.DataFrame(cv_results_C)

# Combine A, B and C
cv_results_ABC = pd.concat(
    [
        cv_results_df,
        cv_results_C_df
    ],
    ignore_index=True
)

metrics_ABC = [
    "Accuracy",
    "Macro_Precision",
    "Macro_Recall",
    "Macro_F1",
    "Fatal_Recall",
    "Fatal_F1",
    "Serious_Recall",
    "Serious_F1",
    "Slight_Recall"
]

abc_cv_summary = (
    cv_results_ABC
    .groupby("Model")[metrics_ABC]
    .agg(["mean", "std"])
    .round(4)
)

print("\nRepeated CV — Final A/B/C Comparison")
print("=" * 110)
print(abc_cv_summary.to_string())

In [ ]:
# Cell 16: Save the final Ethiopia repeated-CV, driver-ablation, feature-importance, and descriptive-analysis tables for reproducible reporting.

from pathlib import Path

table_dir = Path("../outputs/tables")
table_dir.mkdir(parents=True, exist_ok=True)

# 1. Full repeated CV results — all 45 evaluations
cv_results_ABC.to_csv(
    table_dir / "Table_Ethiopia_Repeated_CV_All_Results.csv",
    index=False
)

# 2. Final A/B/C mean ± SD comparison
abc_cv_summary.to_csv(
    table_dir / "Table_Ethiopia_ABC_CV_Summary.csv"
)

# 3. Driver feature ablation analysis
driver_ablation_df.to_csv(
    table_dir / "Table_Ethiopia_Driver_Ablation.csv",
    index=False
)

# 4. Model A feature importance
feature_importance_A.to_csv(
    table_dir / "Table_Ethiopia_ModelA_Feature_Importance.csv",
    index=False
)

# 5. Model B feature importance
feature_importance_B.to_csv(
    table_dir / "Table_Ethiopia_ModelB_Feature_Importance.csv",
    index=False
)

# 6. Driver severity descriptive tables
for feature, table in driver_severity_tables.items():

    safe_name = feature.replace(" ", "_")

    table.to_csv(
        table_dir /
        f"Table_Ethiopia_Severity_by_{safe_name}.csv"
    )

print("Saved Ethiopia tables:")
print("1. Table_Ethiopia_Repeated_CV_All_Results.csv")
print("2. Table_Ethiopia_ABC_CV_Summary.csv")
print("3. Table_Ethiopia_Driver_Ablation.csv")
print("4. Table_Ethiopia_ModelA_Feature_Importance.csv")
print("5. Table_Ethiopia_ModelB_Feature_Importance.csv")
print("6. Five driver-severity descriptive tables")